# 03_04_AdamW_＋_Warmup_＋_Cosine_

コード：`ch03/blocks/03_04_AdamW_＋_Warmup_＋_Cosine_.ipynb`

本文のコードをコードセルで直接実行します。

上からセルを実行してください。ローカルではコード一式と同じPython環境をJupyterのカーネルに選びます。Colabでコード一式が必要な場合は、準備セルで配布ZIPをアップロードします。実行確認の範囲は `VALIDATION_COLAB_20261110.md` を参照してください。


In [ ]:
import subprocess
import sys

# 既に導入済みなら不要な更新をせず、満たしていない依存だけを導入する。
subprocess.run([sys.executable, "-m", "pip", "install", *['numpy>=1.26,<3', 'torch>=2.6,<3', 'torchvision>=0.21,<1', 'mcp==2.3.0', 'PyYAML==6.0.3', 'opentelemetry-sdk==1.45.2', 'opentelemetry-exporter-otlp-proto-http==1.45.2']], check=True)


In [ ]:
import math, torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(0)                      # 再現性：まずこれ

# ---- データ（y = 2x + 1 にノイズ）------------------------------
x = torch.linspace(-1, 1, 100).unsqueeze(1)
y = 2 * x + 1 + 0.05 * torch.randn_like(x)
perm = torch.randperm(100)                # ★ 並び順のまま切ると外挿になる
x, y = x[perm], y[perm]
n_tr = 80
train = DataLoader(TensorDataset(x[:n_tr], y[:n_tr]), batch_size=10, shuffle=True)

model   = nn.Linear(1, 1)
loss_fn = nn.MSELoss()

# ---- ★ weight decay を「重みだけ」に掛ける ---------------------
decay    = [p for p in model.parameters() if p.ndim >  1]   # 行列
no_decay = [p for p in model.parameters() if p.ndim <= 1]   # bias / LayerNorm
opt = torch.optim.AdamW(
    [{"params": decay,    "weight_decay": 0.01},
     {"params": no_decay, "weight_decay": 0.0}],
    lr=3e-2,                              # ★ 課題に対して十分な大きさか要確認
    betas=(0.9, 0.999), eps=1e-8,
)

# ---- スケジュール：Warmup → Cosine（下限つき）------------------
EPOCHS      = 30
total_steps = EPOCHS * len(train)
warmup      = max(1, int(total_steps * 0.1))
MIN_RATIO   = 0.1                         # ★ 0 まで落とさない

def factor(step):
    if step < warmup:
        return (step + 1) / warmup        # ★ +1：1 step 目を lr=0 にしない
    p = (step - warmup) / max(1, total_steps - warmup)
    return MIN_RATIO + (1 - MIN_RATIO) * 0.5 * (1 + math.cos(math.pi * p))

sch = torch.optim.lr_scheduler.LambdaLR(opt, factor)

# ---- 学習ループ ------------------------------------------------
step = 0
for epoch in range(EPOCHS):
    model.train()
    for bx, by in train:
        lr_used = sch.get_last_lr()[0]    # ★ step() の «前» に読む

        opt.zero_grad(set_to_none=True)   # ①
        pred = model(bx)                  # ②
        loss = loss_fn(pred, by)          # ③
        loss.backward()                   # ④
        gnorm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()                        # ⑤ ここで lr_used が効く
        sch.step()                        # 次の step 用に lr を更新
        step += 1

        if step % 50 == 0:
            print(f"step={step:4d} loss={loss.item():.5f} "
                  f"lr={lr_used:.6f} |g|={gnorm:.3f}")

# ---- 検証（勾配を切る・eval にする）----------------------------
model.eval()
with torch.no_grad():
    val = loss_fn(model(x[n_tr:]), y[n_tr:]).item()
print(f"val={val:.6f}  w={model.weight.item():.4f}  b={model.bias.item():.4f}")
# → val=0.004  w=1.989  b=1.001   （目標 2 と 1）
